# Lambdas e streams

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Passar comportamento como valor com uma lambda, como um comparador.
- Montar um pipeline de stream: filtrar, mapear e ordenar, depois uma operação terminal.
- Agrupar e contar com collectors em vez de laços.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma lambda: uma função como valor
2. Lambdas no dia a dia
3. Referências a métodos
4. Um pipeline de stream
5. map, sorted e toList
6. Operações terminais: obtendo uma resposta
7. Agrupando com collectors
8. Streams são preguiçosos e de uso único

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Uma lambda: uma função como valor

Uma **lambda** é uma pequena função escrita como valor: `n -> n * 2` recebe `n` e devolve `n * 2`. Você pode guardá-la numa variável e passá-la para métodos. O tipo dela diz o que recebe e o que devolve:

| Tipo | Recebe → devolve | Chame com |
|---|---|---|
| `Predicate<T>` | um T → verdadeiro ou falso | `test(x)` |
| `Function<T, R>` | um T → um R | `apply(x)` |
| `Consumer<T>` | um T → nada (ele faz algo) | `accept(x)` |

In [ ]:
Predicate<Integer> isEven = n -> n % 2 == 0;
Function<String, Integer> length = s -> s.length();
Consumer<String> shout = s -> IO.println(s.toUpperCase() + "!");

IO.println(isEven.test(4) + " " + isEven.test(7));
IO.println(length.apply("lambda"));
shout.accept("olá");

🧪 **Experimente:** escreva um `Predicate<String>` que diz se uma palavra começa com `"a"`, e teste em duas palavras.

## 2. Lambdas no dia a dia

Muitos métodos recebem uma lambda para saber **o que fazer**: `removeIf` recebe o teste dos itens a tirar, `forEach` o que fazer com cada item, e `sort` como comparar dois itens. Um comparador recebe dois itens e devolve um número negativo, zero ou positivo. `Comparator.reverseOrder()` e `Comparator.comparing(...)` montam os mais comuns para você.

In [ ]:
List<Integer> numbers = new ArrayList<>(List.of(5, 12, 3, 8, 21, 6));
numbers.removeIf(n -> n % 3 == 0);
IO.println(numbers);
numbers.sort(Comparator.reverseOrder());
IO.println(numbers);
numbers.forEach(n -> IO.print(n * 10 + " "));
IO.println();

🧪 **Experimente:** ordene em ordem crescente com a lambda `(a, b) -> Integer.compare(a, b)`. O que `(a, b) -> Integer.compare(b, a)` faz? (Evite `a - b`: estoura para números muito distantes.)

## 3. Referências a métodos

Quando uma lambda só chama um método que já existe, você pode citar o método no lugar: `s -> s.toUpperCase()` vira `String::toUpperCase`, e `x -> System.out.println(x)` vira `System.out::println`. É a mesma coisa, mais curta.

In [ ]:
List<String> words = new ArrayList<>(List.of("pera", "figo", "banana"));
words.replaceAll(String::toUpperCase);       // o mesmo que s -> s.toUpperCase()
words.forEach(System.out::println);          // o mesmo que w -> System.out.println(w)
words.sort(Comparator.comparing(String::length));
IO.println(words);

🧪 **Experimente:** reescreva as duas referências a métodos como lambdas, rode e confira que a saída é a mesma.

## 4. Um pipeline de stream

Um **stream** processa uma coleção como uma linha de montagem: uma **origem** (`lista.stream()`), quantas **etapas intermediárias** quiser, cada uma devolvendo um stream novo, e **uma operação terminal** que produz o resultado.

- `filter(teste)` mantém os itens que passam no teste;
- `map(f)` transforma cada item em outro;
- `sorted()` os põe em ordem;
- `sum()`, `count()`, `toList()`… encerram o pipeline.

Os slides acompanham `[3, 8, 5, 12]` passando por `filter`, `map` e `sum`.

#### 🎞️ Um pipeline de stream, etapa por etapa

![Stream.of(3, 8, 5, 12): o filter mantém os valores acima de 4, o map os dobra para 16, 10, 24, e o sum dá 50](stream-steps.pt-br.svg)

In [ ]:
List<Integer> values = List.of(3, 8, 5, 12);
int result = values.stream()
        .filter(x -> x > 4)
        .mapToInt(x -> x * 2)
        .sum();
IO.println(result);
IO.println(values);   // a lista de origem não mudou

🧪 **Experimente:** tire a linha do `filter` e preveja a soma antes. Depois mantenha só os pares em vez dos maiores que 4.

### ✏️ Exercício: Conte as palavras longas

Conte as palavras com pelo menos `min` letras, com um stream: filter e depois count.

In [ ]:
long countLong(List<String> words, int min) {
    // TODO: words.stream().filter(...).count()
    return 0;
}

void main() {
    check("duas longas", countLong(List.of("a", "stream", "of", "words"), 4), 2L);
    check("nenhuma", countLong(List.of("a", "b"), 3), 0L);
}

<details>
<summary>Ver uma resposta</summary>

```java
long countLong(List<String> words, int min) {
    return words.stream().filter(w -> w.length() >= min).count();
}

void main() {
    check("duas longas", countLong(List.of("a", "stream", "of", "words"), 4), 2L);
    check("nenhuma", countLong(List.of("a", "b"), 3), 0L);
}
```

</details>

## 5. map, sorted e toList

O `map` transforma cada item em outro, até de outro tipo (uma palavra no tamanho dela). O `sorted()` usa a ordem natural, ou um comparador que você passar. O `toList()` junta o resultado numa lista nova. O `distinct()` tira os repetidos, e o `limit(n)` fica com os n primeiros.

In [ ]:
List<Integer> readings = List.of(12, -3, 7, 3, 0, -12);
List<Integer> sizes = readings.stream()
        .map(Math::abs)          // the size of each reading, without its sign
        .distinct()
        .sorted()
        .toList();
IO.println(sizes);
List<String> labels = sizes.stream().map(n -> "#" + n).toList();
IO.println(labels);

🧪 **Experimente:** mova o `distinct()` para antes da linha do `map`: por que 3 e 12 aparecem duas vezes agora? Depois acrescente `.limit(2)`.

### ✏️ Exercício: Grite cada palavra

Devolva uma lista com cada palavra em maiúsculas, com um stream: map e depois toList.

In [ ]:
List<String> shout(List<String> words) {
    // TODO: mapeie cada palavra com w -> w.toUpperCase(Locale.ROOT)
    return List.of();
}

void main() {
    check("duas palavras", shout(List.of("hi", "there")), List.of("HI", "THERE"));
    check("vazio", shout(List.of()), List.of());
}

<details>
<summary>Ver uma resposta</summary>

```java
List<String> shout(List<String> words) {
    return words.stream().map(w -> w.toUpperCase(Locale.ROOT)).toList();   // ROOT: the same in every country
}

void main() {
    check("duas palavras", shout(List.of("hi", "there")), List.of("HI", "THERE"));
    check("vazio", shout(List.of()), List.of());
}
```

</details>

## 6. Operações terminais: obtendo uma resposta

A operação terminal decide o que sai:

| Operação | Devolve |
|---|---|
| `count()` | quantos itens (um `long`) |
| `sum()`, `average()`, `max()` | em streams de números (`mapToInt`, `mapToDouble`) |
| `anyMatch(teste)`, `allMatch(teste)` | `true` ou `false` |
| `toList()` | uma lista |
| `collect(Collectors.joining(", "))` | uma String só |

`average()` e `max()` devolvem um **Optional**, porque um stream vazio não tem média: `orElse(0)` dá um valor reserva.

In [ ]:
List<Double> totals = List.of(120.0, 35.5, 60.0, 15.0);
IO.println(totals.stream().filter(t -> t >= 50).count() + " pedidos grandes");
IO.println("Média: " + totals.stream().mapToDouble(t -> t).average().orElse(0));
IO.println("Algum acima de 100? " + totals.stream().anyMatch(t -> t > 100));
IO.println("Todos acima de 10? " + totals.stream().allMatch(t -> t > 10));
IO.println(totals.stream().map(t -> "R$" + t).collect(Collectors.joining(", ")));

🧪 **Experimente:** calcule a média de uma lista vazia (`List.<Double>of()`) com e sem o `orElse(0)`. Depois ache o maior total com `max()`.

## 7. Agrupando com collectors

`collect(Collectors.groupingBy(chave))` monta um **Map** numa linha só: a chave é calculada para cada item, e cada chave recebe a lista dos itens dela. Passe um segundo collector para obter outra coisa por chave, como `Collectors.counting()`.

In [ ]:
List<String> animals = List.of("gato", "galo", "cão", "cabra", "pato", "abelha");
Map<Character, List<String>> byLetter = animals.stream()
        .collect(Collectors.groupingBy(a -> a.charAt(0)));
IO.println(byLetter);
Map<Character, Long> howMany = animals.stream()
        .collect(Collectors.groupingBy(a -> a.charAt(0), Collectors.counting()));
IO.println(howMany);

🧪 **Experimente:** agrupe pela **última** letra. Depois compare com o laço que você precisaria para montar o mesmo mapa na mão.

## 8. Streams são preguiçosos e de uso único

Duas surpresas. As etapas intermediárias **não fazem nada** até uma operação terminal rodar: um pipeline sem ela nunca roda. E um stream só pode ser usado **uma vez**: para fazer uma segunda pergunta, comece de novo com um `.stream()` novo.

In [ ]:
List<Integer> values = List.of(1, 2, 3);
values.stream().map(v -> {
    IO.println("mapeando " + v);     // nunca impresso: não há operação terminal
    return v * 2;
});

Stream<Integer> once = values.stream().filter(v -> v > 1);
IO.println(once.count());
try {
    IO.println(once.count());        // o segundo uso
} catch (IllegalStateException e) {
    IO.println("Erro: " + e.getMessage());
}

🧪 **Experimente:** acrescente `.toList()` depois do primeiro `map` e rode: agora as linhas "mapeando" aparecem.

## Em resumo

Uma **lambda** é uma pequena função escrita como valor: `t -> t >= 50` recebe `t` e responde verdadeiro ou falso. Um **stream** roda um pipeline sobre uma coleção: uma origem, depois passos como `filter`, `map` e `sorted`, depois **uma operação terminal** que dá o resultado.

```java
void main() {
    List<Double> totals = List.of(120.0, 35.5, 60.0, 15.0);
    double big = totals.stream()
            .filter(t -> t >= 50)
            .mapToDouble(t -> t)
            .sum();
    long small = totals.stream().filter(t -> t < 50).count();
    boolean huge = totals.stream().anyMatch(t -> t > 100);
    IO.println(big + " em pedidos grandes, " + small + " pequenos, algum enorme: " + huge);
}
```

As operações terminais incluem `sum`, `count`, `toList()` e `collect(...)`; `Collectors.groupingBy` e `partitioningBy` montam mapas.

**Cuidado:** um stream só pode ser usado uma vez, e nada roda até a operação terminal. Comece um novo `.stream()` para cada pipeline.

## Leituras

- [Processing Data in Memory Using the Stream API](https://dev.java/learn/api/collections-and-streams/streams/map-filter-reduce/) (dev.java, 9 min): Map, filter e reduce comparados com o mesmo laço, operações intermediárias (devolvem um stream) versus terminais (produzem o resultado), e por que nada roda até a terminal.
- [Writing Your First Lambda Expression](https://dev.java/learn/language/fp/lambdas/first-lambdas/) (dev.java, 9 min): Opcional: a sintaxe (parâmetros) -> corpo, com uma expressão ou um bloco entre chaves. As partes sobre capturar valores locais e serialização são mais do que você precisa agora.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Ordenar com Comparator (lambdas)** · Fácil
- **Quadrados dos pares** · Fácil
- **Estatísticas com streams** · Fácil
- **Arrumar uma lista de nomes** · Médio
- **Agrupar palavras por tamanho** · Médio
- **Aprovado ou reprovado (partitioningBy)** · Médio
- **Os 3 maiores pontuadores** · Difícil
- **Índice de palavras (flatMap)** · Difícil
- Teste: **Relatório da turma** · Difícil